In [380]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [381]:
# BM25Index 实现
import math
from collections import Counter
from typing import Callable, Optional, Any, List, Dict, Tuple


class BM25Index:
    """基于 BM25 的全文检索索引，按词频与逆文档频率对查询与文档打分。"""

    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        """
        初始化 BM25 索引。
        :param k1: 词频饱和参数，控制词频对分数的影响程度
        :param b: 文档长度归一化参数，0 不归一化，1 完全按相对长度归一化
        :param tokenizer: 可选，将文本切分为词列表的函数；默认按非字母数字切分并转小写
        """
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        """默认分词：转小写后按非字母数字切分，过滤空串。"""
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        """添加一篇文档的分词后，更新文档长度与词项文档频（用于后续 IDF）。"""
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        """根据当前文档集合计算每个词项的 IDF（逆文档频率）。"""
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        """计算平均文档长度与 IDF，标记索引已构建。"""
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        """添加一篇文档，文档须为包含 'content' 键的字典，会对 content 分词并更新统计。"""
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("文档的 'content' 必须是字符串。")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def _compute_bm25_score(
        self, query_tokens: List[str], doc_index: int
    ) -> float:
        """对单个文档计算 BM25 分数（查询词在该文档上的加权得分之和）。"""
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query_text: str,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        """
        检索与查询最相关的 k 个文档。
        :param query_text: 查询字符串
        :param k: 返回的最大文档数
        :param score_normalization_factor: 用于将 BM25 原始分转为 [0,1] 的归一化因子（exp(-factor*raw_score)）
        :return: [(文档, 归一化分数), ...]，按归一化分数升序（分数越高越相关）
        """
        if not self.documents:
            return []

        if not isinstance(query_text, str):
            raise TypeError("query_text 必须是字符串。")

        if k <= 0:
            raise ValueError("k 必须是正整数。")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        """返回索引中的文档数量。"""
        return len(self.documents)

    def __repr__(self) -> str:
        """返回索引简要描述（文档数、k1、b、是否已构建）。"""
        return f"BM25Index(文档数={len(self)}, k1={self.k1}, b={self.b}, 已构建={self._index_built})"

In [382]:
with open("./RAG/report_cn.md", "r") as f:
    text = f.read()

In [383]:
# 1. 按章节分块文本
chunks = chunk_by_section(text)

In [384]:
# 2. 创建BM25存储并添加文档
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

In [385]:
# 3. 搜索存储
results = store.search("What happened with INC-2023-Q4-011?", 3)

# 打印结果
for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")

0.2923470957561414 
 第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑 Project Phoenix 的核心系统的稳定性与性能。反复出现的问题，尤其是高峰负载下的 `ERR_MEM_ALLOC_FAIL_0x8007000E` 以及影响数据检索的 `TIMEOUT_QUERY_DB_0xDEADBEEF`，被列为优先处理项，对应事件成本为 INC-2023-Q4-01 
----

0.3635797951702454 
 第十节：网络安全分析——事件响应报告：INC-2023-Q4-011

网络安全运营中心成功遏制并修复了编号为 `INC-2023-Q4-011` 的定向入侵尝试。威胁情报显示，该活动与 `ShadowNet Syndicate` 威胁行为者组织的战术、技术与程序相符。初始访问通过针对财务部门人员的鱼叉式钓鱼邮件获得，可能意在获取与第三节（财务分析）相关的数据。端点检测与响应（EDR）系统在工作站 
----

